In [1]:
!pip install transformers datasets torch scikit-learn seqeval

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.0 MB 7.6 MB/s eta 0:00:02
     --------- ------------------------------ 2.9/12.0 MB 7.6 MB/s eta 0:00:02
     --------------- ------------------------ 4.7/12.0 MB 8.0 MB/s eta 0:00:01
     --------------------- ------------------ 6.6/12.0 MB 8.3 MB/s eta 0:00:01
     ------------------------------ --------- 9.2/12.0 MB 8.9 MB/s eta 0:00:01
     -------------------------------------- - 11.5/12.0 MB 9.3 MB/s eta 0:00:01
     ---------------------------------------- 12.0/12.0 MB 9.0 MB/s eta 0:00:00
  Using cached seqeval-1.2.2-py3-none-any.whl
     ---------------------------------------- 0.0/566.1 kB ? eta -:--:--
     -------------------------------------- 566.1/566.1 kB 7.0 MB/s eta 0:00:00
     ------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
!pip install transformers[torch]

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable


ERROR: Invalid requirement: 'accelerate=0.26.1': Expected end or semicolon (after name and no valid version specifier)
    accelerate=0.26.1
              ^
Hint: = is not a valid operator. Did you mean == ?


In [2]:
import json
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict, ClassLabel, Sequence
import transformers
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
from seqeval.metrics import classification_report, f1_score

# --- 配置参数 ---
JSON_FILE = 'annotated_data_100.json'  # 包含您900条标注数据的文件
MODEL_NAME = "bert-base-chinese"       # 预训练模型
OUTPUT_DIR = "./absa_baseline_model"   # 模型训练后保存的目录
TEST_SIZE = 0.2                        # 划分多少数据作为测试集 (例如 20%)
RANDOM_SEED = 42                       # 保证数据划分可复现
NUM_TRAIN_EPOCHS = 3                   # 训练轮数 (基线模型可以少跑几轮)
BATCH_SIZE = 16                        # 根据您的GPU显存调整

# --- 1. 加载和预处理数据 ---

print(f"Loading data from {JSON_FILE}...")
try:
    with open(JSON_FILE, 'r', encoding='utf-8') as f:
        full_data = json.load(f)
        # 确保是列表
        if not isinstance(full_data, list):
            raise ValueError("JSON file root is not a list.")
except Exception as e:
    print(f"Error loading or parsing JSON file: {e}")
    exit()

print(f"Loaded {len(full_data)} annotated comments.")

# --- 2. 转换数据为序列标注格式 (BIOES) ---
# 定义标签体系: O (Outside), B-ASP-POS, I-ASP-POS, E-ASP-POS, S-ASP-POS, ... NEG, NEU
# 这是一个 *示例性* 的转换函数，可能需要调试和优化
label_list = ["O", 
              "B-ASP-POS", "I-ASP-POS", "E-ASP-POS", "S-ASP-POS",
              "B-ASP-NEG", "I-ASP-NEG", "E-ASP-NEG", "S-ASP-NEG",
              "B-ASP-NEU", "I-ASP-NEU", "E-ASP-NEU", "S-ASP-NEU"]
label_map = {label: i for i, label in enumerate(label_list)}
num_labels = len(label_list)

def convert_to_bioes(example, tokenizer):
    tokens = []
    labels = []
    words = list(example['comment_text']) # 简单的按字切分

    # 初始化所有字的标签为 'O'
    word_labels = ['O'] * len(words)

    annotations = example.get('annotations', [])
    # 按照 aspect 在原文中出现的起始位置排序，防止标签覆盖出错
    annotations.sort(key=lambda x: example['comment_text'].find(x['aspect']))

    processed_indices = set() # 记录已经被标注过的字索引

    for ann in annotations:
        aspect = ann['aspect']
        sentiment = ann['sentiment'][:3].upper() # POS, NEG, NEU
        start_index = example['comment_text'].find(aspect)

        if start_index == -1:
            print(f"Warning: Aspect '{aspect}' not found in text: '{example['comment_text']}'")
            continue

        end_index = start_index + len(aspect)

        # 检查是否有重叠标注 (简单处理：跳过重叠的)
        current_indices = set(range(start_index, end_index))
        if any(idx in processed_indices for idx in current_indices):
            print(f"Warning: Overlapping annotation skipped for aspect '{aspect}' in text: '{example['comment_text']}'")
            continue


        if len(aspect) == 1:
            word_labels[start_index] = f"S-ASP-{sentiment}"
        else:
            word_labels[start_index] = f"B-ASP-{sentiment}"
            for i in range(start_index + 1, end_index -1):
                 word_labels[i] = f"I-ASP-{sentiment}"
            word_labels[end_index - 1] = f"E-ASP-{sentiment}"

        processed_indices.update(current_indices)


    # --- Tokenize 并对齐标签 ---
    # 使用is_split_into_words=True，因为我们已经按字切分了
    tokenized_inputs = tokenizer(words, truncation=True, is_split_into_words=True)

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
        if word_idx is None: # Special tokens like [CLS], [SEP]
            label_ids.append(-100) # -100 ID tells CrossEntropyLoss to ignore these tokens
        elif word_idx != previous_word_idx: # Only label the first token of a given word.
            label_index = label_map.get(word_labels[word_idx], label_map["O"]) # Default to 'O' if label not found
            label_ids.append(label_index)
        else:
            # We label subsequent tokens of the same word with -100 as well
            # Alternatively, you could propagate the label (e.g., B-tag, I-tag, I-tag...)
            label_ids.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs


# --- 3. 加载Tokenizer ---
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# --- 4. 准备Dataset ---
print("Converting data to BIOES format and tokenizing...")
# 稍微有点慢，因为需要处理每条数据
processed_data = []
for item in full_data:
   processed_item = convert_to_bioes(item, tokenizer)
   # 为了创建Dataset，我们需要确保字段名一致
   # Trainer期望 'input_ids', 'token_type_ids', 'attention_mask', 'labels'
   processed_data.append({
       'input_ids': processed_item['input_ids'],
       'token_type_ids': processed_item['token_type_ids'],
       'attention_mask': processed_item['attention_mask'],
       'labels': processed_item['labels']
   })


# 将处理好的数据转换为 Hugging Face Dataset
hf_dataset = Dataset.from_list(processed_data)

# --- 5. 划分训练集和测试集 ---
print(f"Splitting data into train/test sets (test_size={TEST_SIZE})...")
train_test_split_dict = hf_dataset.train_test_split(test_size=TEST_SIZE, seed=RANDOM_SEED)
dataset_dict = DatasetDict({
    'train': train_test_split_dict['train'],
    'test': train_test_split_dict['test']
})
print(dataset_dict)


# --- 6. 加载模型 ---
print(f"Loading model {MODEL_NAME} for token classification...")
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# --- 7. 定义训练参数 ---
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch", # 每个epoch结束后评估一次
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    weight_decay=0.01,
    save_strategy="epoch", # 每个epoch保存一次模型
    load_best_model_at_end=True, # 训练结束后加载最好的模型
    # push_to_hub=False, # 如果你想上传到Hugging Face Hub，设为True并登录
)

# --- 8. 定义评估指标 ---
# seqeval 需要标签的字符串形式，而不是ID
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    report = classification_report(true_labels, true_predictions, output_dict=True)
    
    # 只取几个关键指标
    results = {
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"],
       #"accuracy": report["accuracy"]
    }
    return results

# --- 9. 初始化Trainer ---
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# --- 10. 开始训练 ---
print("Starting training...")
trainer.train() # <--- 取消这行注释以开始训练

# --- 11. 评估模型 (如果训练了) ---
print("Evaluating model...")
eval_results = trainer.evaluate() # <--- 取消这行注释以评估
print(eval_results)

# --- 12. 保存模型 (Trainer会自动保存) ---
print(f"Baseline model training setup complete. Model would be saved to {OUTPUT_DIR}")
print("To run training, uncomment the 'trainer.train()' line.")
print("After training, uncomment 'trainer.evaluate()' to see the performance on the test set.")

Loading data from annotated_data_100.json...
Loaded 1000 annotated comments.
Loading tokenizer for bert-base-chinese...
Converting data to BIOES format and tokenizing...
Splitting data into train/test sets (test_size=0.2)...
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 800
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
})
Loading model bert-base-chinese for token classification...


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\12611\AppData\Local\Temp\ipykernel_39496\3019227334.py:191: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.275674,0.581216,0.750469,0.652330
2,No log,0.104432,0.916845,0.943715,0.929766
3,No log,0.087157,0.932316,0.956848,0.944239


C:\Users\12611\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\12611\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Evaluating model...


C:\Users\12611\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.08715718239545822, 'eval_precision': 0.9323156910040504, 'eval_recall': 0.9568480300187617, 'eval_f1': 0.944239173579222, 'eval_runtime': 12.0545, 'eval_samples_per_second': 16.591, 'eval_steps_per_second': 1.078, 'epoch': 3.0}
Baseline model training setup complete. Model would be saved to ./absa_baseline_model
To run training, uncomment the 'trainer.train()' line.
After training, uncomment 'trainer.evaluate()' to see the performance on the test set.
